In [ ]:
# --- Imports ---
from config import CONFIG
from preprocessing import load_and_prepare
from training import run_training
from plotting import save_metrics, plot_confusion_matrix, plot_average_roc

print("Experiment settings:")
for k, v in CONFIG.items():
    if k != "split_files":
        print(f"{k}: {v}")
print(f"split_file: {CONFIG['split_files'][CONFIG['split_strategy']]}")

In [ ]:
# --- Preprocessing ---
print("Loading and preprocessing data...")
data_dict = load_and_prepare(CONFIG)
print("Preprocessing complete.")

In [ ]:
# --- Training ---
print("Training with cross-validation...")
results = run_training(CONFIG, data_dict)
print("Training complete.")

In [ ]:
# --- Save metrics + plots ---
print("Step 3: Saving results...")
save_metrics(
    results["metrics_df"],results["mean_metrics"],results["std_metrics"],
    CONFIG,results["fold_reports"],results["int_to_label"])

plot_confusion_matrix(results["cumulative_cm"], results["int_to_label"], CONFIG)
plot_average_roc(results["y_true_folds"], results["y_prob_folds"], results["int_to_label"], CONFIG)
print("Results saved.")

print("Pipeline finished successfully!") 

### Merge all the per metrics moa into 1 single csv file

In [ ]:
import os
import pandas as pd

# Path to your results folder
base_path = "/home/gauss/Documents/Project 2: Combining Molecular and Imagining Data for MOA Prediction/Image model/Image-based model/results/"

# Define categories to group by
categories = [
    "as_is",
    "merge_tk_egfr_vegfr",
    "merge_tk_egfr_keep_vegfr",
    "merge_tk_vegfr_keep_egfr",
    "merge_all_kinases",
    "merge_st_vs_tk"
]

# Get all folders that start with "U2OS_"
folders = [f for f in os.listdir(base_path) if f.startswith("HEPG2_") and os.path.isdir(os.path.join(base_path, f))]

# Process each category separately
for category in categories:
    merged = None
    # Select only folders that contain the category keyword
    category_folders = [f for f in folders if category in f]
    
    for folder in category_folders:
        file_path = os.path.join(base_path, folder, "per_moa_metrics.csv")
        if os.path.exists(file_path):
            df = pd.read_csv(file_path)
            df = df.set_index("moa")
            df.columns = pd.MultiIndex.from_product([[folder], df.columns])  # folder name on top
            
            if merged is None:
                merged = df
            else:
                merged = merged.join(df, how="outer")
    
    # Save result for this category if we found any data
    if merged is not None:
        merged = merged.reset_index()
        output_path = os.path.join(base_path, f"HEPG2_merged_{category}.csv")
        merged.to_csv(output_path, index=False)
        print(f"Saved: {output_path}")
    else:
        print(f"No data found for category: {category}")
